In [ ]:
# 1. Install Libraries
!pip install -q transformers accelerate scikit-learn

import os
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive

# Fix seed for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [ ]:
#cell 2

from huggingface_hub import login
login()

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
model.eval()

# Punctuation ID Mapping
PUNCT_MAP = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1]
}
EOS_ID = tokenizer.eos_token_id
print(f"Punctuation IDs: {PUNCT_MAP}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Punctuation IDs: {'COMMA': 11, 'PERIOD': 13, 'QMARK': 30}


In [ ]:
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. User-provided paths and loading code
BASE_PATH = "/content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2"
VAL_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_validation.Y.txt")
TEST_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_test.Y.txt")

assert os.path.exists(VAL_Y_PATH), f"Not found: {VAL_Y_PATH}"

print(f"Loading file: {VAL_Y_PATH}")
with open(VAL_Y_PATH, "r", encoding="utf-8") as f:
    val_y_list = [line.strip() for line in f if line.strip()]

with open(TEST_Y_PATH, "r", encoding="utf-8") as f:
    test_y_list = [line.strip() for line in f if line.strip()]

print("num val samples:", len(val_y_list))
print("example:", val_y_list[0])

Mounted at /content/drive
Loading file: /content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2/iwslt2017_en_validation.Y.txt
num val samples: 1501
example: Last year I showed these two slides so that demonstrate that the arctic ice cap, which for most of the last three million years has been the size of the lower 48 states, has shrunk by 40 percent.


In [ ]:
# [Cell: Inspect Baseline Generation Output]
import torch
import pandas as pd
import time
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import re
import numpy as np

def parse_sentence_to_boundaries(text):
    tokens = text.strip().split()
    boundaries = []
    for tok in tokens:
        word = tok
        label = "O"
        if tok.endswith(","): label = "COMMA"; word = tok[:-1]
        elif tok.endswith("."): label = "PERIOD"; word = tok[:-1]
        elif tok.endswith("?"): label = "QMARK"; word = tok[:-1]
        word = re.sub(r'[\"\\'\(\)]', '', word)
        if word: boundaries.append({"word": word, "label": label})
    return boundaries

# --- 1. Select a sentence to inspect ---
# Select an index further back to find sentences that clearly contain punctuation.
# Try changing to your desired index.
SAMPLE_IDX = 52

if SAMPLE_IDX >= len(val_y_list):
    print(f"Error: Index {SAMPLE_IDX} is out of range.")
else:
    # Get Gold (True) sentence
    gold_sentence = val_y_list[SAMPLE_IDX]

    # Create Input sentence (remove punctuation)
    boundaries = parse_sentence_to_boundaries(gold_sentence)
    input_words = [b['word'] for b in boundaries]
    input_text = " ".join(input_words)

    print(f"=== [Debug Analysis] Sample Index: {SAMPLE_IDX} ===")
    print(f"▶ Original (Gold): {gold_sentence}")
    print(f"▶ Input (Unpunctuated): {input_text}")

    # Prompt Template (User Provided)
    PROMPT_TEMPLATE = """You are a punctuation restoration tool.
Rules:
Insert punctuation only (comma, period, question mark).
Do not add or remove words.
Keep the original word order.
Return exactly one line: the punctuated sentence.
Text: {}
Output:"""

    # --- 2. Construct and Generate Prompt ---
    formatted_prompt = PROMPT_TEMPLATE.format(input_text)

    print(f"\n[Prompt Sent to Model]")
    print("-" * 40)
    print(formatted_prompt)
    print("-" * 40)

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    # Start Generation
    print("\nGenerating response...", end="")
    start_t = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=len(inputs.input_ids[0]) + 50, # Sufficient margin
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    print(f" Done ({time.time() - start_t:.2f}s)")

    # --- 3. Decode Results ---
    # Remove the prompt part and only view the generated text.
    input_len = inputs.input_ids.shape[1]
    generated_ids = outputs[0][input_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"\n[Model Output]")
    print("=" * 40)
    print(generated_text)
    print("=" * 40)

    # --- 4. Instant Analysis ---
    clean_gen = generated_text.strip()
    clean_input = input_text.strip()

    if clean_gen == clean_input:
        print(">> Analysis: The model copied the input text exactly without adding any punctuation. (Instruction Following Failure)")
    elif len(clean_gen) < len(clean_input):
        print(">> Analysis: The generated text is shorter than the original. (Possible word omission/truncation)")
    else:
        print(">> Analysis: The model attempted to make changes. Visually compare the results above.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== [Debug Analysis] Sample Index: 52 ===
▶ Original (Gold): This, all over the country, is the second largest waste stream in America.
▶ Input (Unpunctuated): This all over the country is the second largest waste stream in America

[Prompt Sent to Model]
----------------------------------------
You are a punctuation restoration tool.
Rules:
Insert punctuation only (comma, period, question mark).
Do not add or remove words.
Keep the original word order.
Return exactly one line: the punctuated sentence.
Text: This all over the country is the second largest waste stream in America
Output:
----------------------------------------

Generating response... Done (1.73s)

[Model Output]
 This, all, over, the, country, is, the, second, largest, waste, stream, in, America.

>> 분석: 모델이 무언가 변경을 시도했습니다. 위 결과를 눈으로 비교해보세요.


In [ ]:
# [Cell: Baseline Experiment - Prompt-based Generation]
import torch
import pandas as pd
import time
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import re
import numpy as np

# --- 1. Configuration ---
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

# Prompt Template (User Provided)
PROMPT_TEMPLATE = """You are a punctuation restoration tool.
Rules:
Insert punctuation only (comma, period, question mark).
Do not add or remove words.
Keep the original word order.
Return exactly one line: the punctuated sentence.
Text: {}
Output:"""

LABELS_ORDER = ["O", "COMMA", "PERIOD", "QMARK"]

def parse_sentence_to_boundaries(text):
    tokens = text.strip().split()
    boundaries = []
    for tok in tokens:
        word = tok
        label = "O"
        if tok.endswith(","): label = "COMMA"; word = tok[:-1]
        elif tok.endswith("."): label = "PERIOD"; word = tok[:-1]
        elif tok.endswith("?"): label = "QMARK"; word = tok[:-1]
        word = re.sub(r'[\"\\'\(\)]', '', word)
        if word: boundaries.append({"word": word, "label": label})
    return boundaries

def get_predicted_labels_from_generation(original_boundaries, generated_text):
    """
    Function to extract labels by aligning generated text with original words.
    If the generative model changes or omits words, it is treated as 'O' or incorrect.
    """
    gen_tokens = generated_text.strip().split()
    pred_labels = []

    gen_idx = 0
    max_gen_idx = len(gen_tokens)

    for item in original_boundaries:
        orig_word = item['word']
        found = False

        # Find original word in generated text (from current position)
        # (Loosely compare using lower() in case the generative model slightly changed words)
        search_limit = min(gen_idx + 5, max_gen_idx) # Don't look too far

        for k in range(gen_idx, search_limit):
            gen_tok = gen_tokens[k]
            # Generated word with punctuation removed
            clean_gen_word = re.sub(r'[,.\?]', '', gen_tok)

            # If words are deemed to match
            if clean_gen_word.lower() == orig_word.lower():
                found = True
                gen_idx = k + 1 # Update next search position

                # Check punctuation
                if gen_tok.endswith(","): pred_labels.append("COMMA")
                elif gen_tok.endswith("."): pred_labels.append("PERIOD")
                elif gen_tok.endswith("?"): pred_labels.append("QMARK")
                else: pred_labels.append("O")
                break

        if not found:
            # If the model omitted words (Hallucination) or wrote them too differently -> treat as 'O' (incorrect)
            pred_labels.append("O")

    return pred_labels

# --- 2. Baseline Evaluation Loop ---
print("Starting Baseline Evaluation (Prompt-based Generation)")
print("Note: This performs autoregressive generation and may be slow.")

all_golds = []
all_preds = []
total_processed_words = 0
start_time = time.time()

# Set to iterate through the entire test_y_list.
eval_dataset = test_y_list

for sent_idx, y_true in enumerate(tqdm(eval_dataset)):
    boundaries = parse_sentence_to_boundaries(y_true)
    if not boundaries: continue

    # 1. Restore original text (unpunctuated version) - Simulate ASR output
    input_words = [b['word'] for b in boundaries]
    input_text = " ".join(input_words)
    total_processed_words += len(input_words)

    # 2. Construct Prompt
    formatted_prompt = PROMPT_TEMPLATE.format(input_text)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    # 3. Generation
    # max_new_tokens: Set with a bit more margin than the input length
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=len(inputs.input_ids[0]) + 50,
            do_sample=False, # Deterministic (Greedy Search)
            pad_token_id=tokenizer.eos_token_id
        )

    # 4. Decode Results (Extract only the generated part, excluding the prompt)
    # Slice by input_ids length and decode only the latter part
    input_len = inputs.input_ids.shape[1]
    generated_ids = outputs[0][input_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # 5. Extract and Align Labels
    preds = get_predicted_labels_from_generation(boundaries, generated_text)
    golds = [b['label'] for b in boundaries]

    # Prevent length mismatch (prevent potential errors)
    min_len = min(len(preds), len(golds))
    all_preds.extend(preds[:min_len])
    all_golds.extend(golds[:min_len])

# --- 3. Results Report ---
end_time = time.time()
elapsed = end_time - start_time
wps = total_processed_words / elapsed

print(f"\n[Baseline Results | Prompt-based Generation]")
print(f"Inference Speed: {wps:.2f} words/s")
print(f"Total Processed Words: {total_processed_words}")
print(f"Total Execution Time: {elapsed:.2f}s")
print("-" * 60)
print(classification_report(all_golds, all_preds, labels=LABELS_ORDER, zero_division=0, digits=3))

print("\nConfusion Matrix")
cm = confusion_matrix(all_golds, all_preds, labels=LABELS_ORDER)
df_cm = pd.DataFrame(cm, index=[f"True_{l}" for l in LABELS_ORDER], columns=[f"Pred_{l}" for l in LABELS_ORDER])
print(df_cm)

Starting Baseline Evaluation (Prompt-based Generation)
Note: This performs autoregressive generation and may be slow.


100%|██████████| 10799/10799 [1:09:31<00:00,  2.59it/s]



[Baseline Results | Prompt-based Generation]
Inference Speed: 44.18 words/s
Total Processed Words: 184280
Total Execution Time: 4171.12s
------------------------------------------------------------
              precision    recall  f1-score   support

           O      0.929     0.989     0.958    160198
       COMMA      0.318     0.110     0.164     12999
      PERIOD      0.922     0.824     0.870     10172
       QMARK      0.980     0.158     0.272       911

    accuracy                          0.914    184280
   macro avg      0.787     0.520     0.566    184280
weighted avg      0.886     0.914     0.894    184280


Confusion Matrix
             Pred_O  Pred_COMMA  Pred_PERIOD  Pred_QMARK
True_O       158399        1714           84           1
True_COMMA    11454        1435          109           1
True_PERIOD     620        1174         8377           1
True_QMARK       58         195          514         144
